In [1]:
import os
os.environ["BITSANDBYTES_NOWELCOME"] = "0"
os.environ["DISABLE_TRITON"] = "0"

In [2]:
# !pip uninstall bitsandbytes -y
# !pip install bitsandbytes==0.41.1

In [3]:
#Dataset

In [4]:
from datasets import load_dataset

alpaca = load_dataset("tatsu-lab/alpaca", split="train[:20000]")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train[:7000]")
#oasst = load_dataset("OpenAssistant/oasst1", split="train[:2000]")

In [5]:
def format_alpaca(example):
    return {
        "instruction": example["instruction"],
        "input": example.get("input", ""),
        "output": example["output"]
    }

def format_dolly(example):
    return {
        "instruction": example["instruction"],
        "input": example.get("context", ""),
        "output": example["response"]
    }

# def format_oasst(example):
#     return {
#         "instruction": example["text"],
#         "input": "",
#         "output": example["text"]
#     }

In [6]:
data = []

data += [format_alpaca(x) for x in alpaca]
data += [format_dolly(x) for x in dolly]
#data += [format_oasst(x) for x in oasst]

In [7]:
import json

with open("final_dataset.json", "w") as f:
    json.dump(data, f, indent=2)

In [8]:
#QloRA 

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig
from trl import SFTTrainer
from peft import prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

In [13]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="final_dataset.json", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [14]:
def format_prompt(example):
    instruction = example['instruction']
    input_text = example['input']

    if input_text.strip() != "":
        prompt = f"<s>[INST] {instruction}\n{input_text} [/INST] {example['output']} </s>"
    else:
        prompt = f"<s>[INST] {instruction} [/INST] {example['output']} </s>"

    return prompt

dataset = dataset.map(lambda x: {"text": format_prompt(x)})

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

In [15]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",   # automatically uses GPU
    torch_dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/home/jyothish/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


In [16]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [31]:
train_test = dataset["train"].train_test_split(test_size=0.2)

train_dataset = train_test["train"]
eval_dataset = train_test["test"]

In [35]:
training_args = TrainingArguments(
    output_dir="./qlora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,

    logging_steps=100,
    
    evaluation_strategy="steps",   # 🔥 enable evaluation
    eval_steps=100,                # evaluate every 100 steps
    
    save_strategy="steps",         # save checkpoints
    save_steps=100,
    
    load_best_model_at_end=True,   # 🔥 MOST IMPORTANT
    metric_for_best_model="eval_loss",
    greater_is_better=False,       # lower loss = better
    
    save_total_limit=2,            # keep only best + last
    
    report_to="none"
)

In [36]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,   # 🔥 IMPORTANT
    peft_config=lora_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512
)

/home/jyothish/torch_env/lib/python3.10/site-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [37]:
trainer.train()

Step,Training Loss,Validation Loss
100,1.694000,1.540261
200,1.393600,1.339248
300,1.303100,1.333684
400,1.307900,1.325865
500,1.313500,1.325559
600,1.317500,1.317559
700,1.269600,1.317725
800,1.307900,1.312186
900,1.267600,1.311997
1000,1.277100,1.311534


/home/jyothish/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/jyothish/torch_env/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/jyothish/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. 

TrainOutput(global_step=4860, training_loss=1.259397676255968, metrics={'train_runtime': 45734.0803, 'train_samples_per_second': 0.85, 'train_steps_per_second': 0.106, 'total_flos': 2.664077310907515e+17, 'train_loss': 1.259397676255968, 'epoch': 2.0})

In [38]:
trainer.save_model("qlora-finetuned")
tokenizer.save_pretrained("qlora-finetuned")

/home/jyothish/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('qlora-finetuned/tokenizer_config.json',
 'qlora-finetuned/special_tokens_map.json',
 'qlora-finetuned/tokenizer.model',
 'qlora-finetuned/added_tokens.json')

In [36]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_path = "qlora-finetuned"

# model = AutoModelForCausalLM.from_pretrained(model_path)
# tokenizer = AutoTokenizer.from_pretrained(model_path)

# prompt = """### Instruction:
# Explain machine learning

# ### Response:
# """

# inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens=100)
# print(tokenizer.decode(outputs[0]))

In [37]:
#RAG

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "qlora-finetuned"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16
).to("cuda")
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_path)

/home/jyothish/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
import json
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Load JSON
with open("final_dataset.json", "r") as f:
    data = json.load(f)

# Convert to documents
documents = []
for item in data:
    text = f"{item['instruction']} {item['output']}"
    documents.append(Document(page_content=text))

# Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)

# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

# Retriever ✅
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_69079/1670108454.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/home/jyothish/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [22]:
def create_prompt(query):
    return f"""<s>[INST]
You are a technical assistant.

Answer the question precisely.
Do NOT change the question.
Do NOT simplify unless asked.

Question:
{query}
[/INST]"""

In [23]:
def rag_pipeline(query):
    # Step 1: Retrieve context
    docs = retriever.invoke(query)   # updated method
    context = " ".join([doc.page_content for doc in docs])

    # Step 2: Create prompt
    prompt = create_prompt(query, context)

    # Step 3: Tokenize
    device = model.device
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Step 4: Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,              # ✅ FIXED
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.2
    )

    # Step 5: Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("[/INST]")[-1].strip()

    return response

In [35]:
prompt = """<s>[INST]
Explain deep learning in a strictly technical manner.
Answer in EXACTLY 4 sentences.
Stop after 4 sentences.
[/INST]"""
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]
Explain deep learning in a strictly technical manner.
Answer in EXACTLY 4 sentences.
Stop after 4 sentences.
[/INST] Deep learning is a subset of machine learning that uses artificial neural networks to learn from data. It is based on the idea that the brain is made up of a network of neurons that are connected to each other and that these connections can be used to learn from data. Deep learning algorithms use multiple layers of neurons to process data and learn from it. The layers are connected to each other and the output of one layer is used as the input for the next layer. This allows the network to learn


In [8]:
# import gradio as gr

# def chat(query):
#     return rag_pipeline(query)

# interface = gr.Interface(
#     fn=chat,
#     inputs="text",
#     outputs="text",
#     title="RAG + QLoRA Assistant"
# )

# interface.launch()